In [ ]:
from openai import OpenAI
from utils import LLM
import pydantic as pt

In [2]:
client = OpenAI()

In [3]:
resp = client.responses.create(
    model=LLM.PRE_NANO, input="Write a one-sentence bedtime story about a unicorn."
)

In [4]:
print(type(resp), isinstance(resp, pt.BaseModel))

<class 'openai.types.responses.response.Response'> True


In [5]:
for attr in dir(resp):
    if attr.startswith("_"):
        continue
    print(attr, type(getattr(resp, attr)))

construct <class 'method'>
copy <class 'method'>
created_at <class 'float'>
dict <class 'method'>
error <class 'NoneType'>
from_orm <class 'method'>
id <class 'str'>
incomplete_details <class 'NoneType'>
instructions <class 'NoneType'>
json <class 'method'>
max_output_tokens <class 'NoneType'>
metadata <class 'dict'>
model <class 'str'>
model_computed_fields <class 'dict'>
model_config <class 'dict'>
model_construct <class 'method'>
model_copy <class 'method'>
model_dump <class 'method'>
model_dump_json <class 'method'>
model_extra <class 'dict'>
model_fields <class 'dict'>
model_fields_set <class 'set'>
model_json_schema <class 'method'>
model_parametrized_name <class 'method'>
model_post_init <class 'method'>
model_rebuild <class 'method'>
model_validate <class 'method'>
model_validate_json <class 'method'>
model_validate_strings <class 'method'>
object <class 'str'>
output <class 'list'>
output_text <class 'str'>
parallel_tool_calls <class 'bool'>
parse_file <class 'method'>
parse_o

/tmp/ipykernel_77059/3004206166.py:4: PydanticDeprecatedSince211: Accessing this attribute on the instance is deprecated, and will be removed in Pydantic V3. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  print(attr, type(getattr(resp, attr)))


In [6]:
resp.model_dump()

{'id': 'resp_68007cde2a4481919c7bbd231a69f2bc0b8bd2ae4d7d4332',
 'created_at': 1744862430.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-4.1-nano-2025-04-14',
 'object': 'response',
 'output': [{'id': 'msg_68007cde7a14819183449c22cf840b3a0b8bd2ae4d7d4332',
   'content': [{'annotations': [],
     'text': 'In a peaceful forest under the twinkling stars, a gentle unicorn lullabies the moon to sleep with her shimmering, melodic voice.',
     'type': 'output_text'}],
   'role': 'assistant',
   'status': 'completed',
   'type': 'message'}],
 'parallel_tool_calls': True,
 'temperature': 1.0,
 'tool_choice': 'auto',
 'tools': [],
 'top_p': 1.0,
 'max_output_tokens': None,
 'previous_response_id': None,
 'reasoning': {'effort': None, 'generate_summary': None, 'summary': None},
 'status': 'completed',
 'text': {'format': {'type': 'text'}},
 'truncation': 'disabled',
 'usage': {'input_tokens': 18,
  'input_tokens_details': {'cached_tokens': 

In [8]:
print(resp.output_text)

In a peaceful forest under the twinkling stars, a gentle unicorn lullabies the moon to sleep with her shimmering, melodic voice.


In [13]:
messages = [
    {"role": "user", "content": "What teams are playing in this image?"},
    {
        "role": "user",
        "content": [
            {
                "type": "input_image",
                "image_url": "https://upload.wikimedia.org/wikipedia/commons/3/3b/LeBron_James_Layup_%28Cleveland_vs_Brooklyn_2018%29.jpg",
            }
        ],
    },
]

In [14]:
resp = client.responses.create(
    model=LLM.PRE_FAST_MINI,
    input=messages,
)
print(resp.output_text)

The teams playing in this image are the Cleveland Cavaliers and the Brooklyn Nets.


In [15]:
resp = client.responses.create(
    model=LLM.PRE_FAST_MINI,
    tools=[{"type": "web_search_preview"}],
    input="What was a positive news story from today?",
)
print(resp.output_text)

As of April 17, 2025, a positive news story highlights the Netherlands' innovative approach to environmental conservation. A unique livestream has gained popularity, allowing viewers to actively participate in helping fish. This initiative has resonated with millions, showcasing the potential of interactive media in promoting ecological awareness and engagement. ([globalgoodnews.com](https://www.globalgoodnews.com/index.html?utm_source=openai)) 


In [16]:
stream = client.responses.create(
    model=LLM.PRE_FAST_MINI,
    input=[{"role": "user", "content": "Say 'double bubble bath' ten times fast."}],
    stream=True,
)
for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='resp_68007e51de508191a524e7553e958d7e0074bf6ecde59dcc', created_at=1744862801.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_output_tokens=None, previous_response_id=None, reasoning=Reasoning(effort=None, generate_summary=None, summary=None), status='in_progress', text=ResponseTextConfig(format=ResponseFormatText(type='text')), truncation='disabled', usage=None, user=None, service_tier='auto', store=True), type='response.created')
ResponseInProgressEvent(response=Response(id='resp_68007e51de508191a524e7553e958d7e0074bf6ecde59dcc', created_at=1744862801.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, max_o

In [17]:
from agents import Agent, Runner
import asyncio

In [18]:
spanish_agent = Agent(name="Spanish agent", instructions="You only speak Spanish")
english_agent = Agent(name="English agent", instructions="You only speak English")
triage_agent = Agent(
    name="Triage agent",
    instructions="Handoff to the appropriate agent based on the language of the request.",
    handoffs=[spanish_agent, english_agent],
)

In [19]:
result = await Runner.run(triage_agent, input="Hola, ¿cómo estás?")
print(result.final_output)

¡Hola! Estoy bien, gracias. ¿Y tú, cómo estás?
